# Lab 08 — Shor 9-Qubit Code (Bit-Flip + Phase-Flip)

The Shor code concatenates the phase-flip code (outer) with the bit-flip
code (inner). It corrects **any** single-qubit error: $X$, $Z$, or $Y \propto XZ$.

| Qubits | Role |
|---|---|
| `q[0..8]` | data qubits in three blocks: (0,1,2), (3,4,5), (6,7,8) |
| `a[0..5]` | bit-flip syndrome ancillas, two per block |
| `a[6..7]` | phase-flip syndrome ancillas |
| `s0, s1, s2` | bit-flip syndromes of blocks 1, 2, 3 |
| `p` | phase-flip syndrome |

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters

In [ ]:
ALPHA, BETA = 0.8, 0.6   # input state alpha|0> + beta|1> (normalized automatically)
X_ERROR = 2              # qubit 0-8 for a bit-flip error, or None
Z_ERROR = 6              # qubit 0-8 for a phase-flip error, or None

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

from common.qlab import (normalize, format_state, print_sparse_state,
                         get_snapshot, qubit_state, fidelity, save_circuit)

for name, value in (("X_ERROR", X_ERROR), ("Z_ERROR", Z_ERROR)):
    if value is not None and value not in range(9):
        raise ValueError(f"{name} must be a qubit number from 0 to 8, or None.")

alpha, beta = normalize(ALPHA, BETA)
backend = AerSimulator(method="statevector")

print("ORIGINAL QUANTUM STATE:", format_state(alpha, beta))
print("X error:", "None" if X_ERROR is None else f"q[{X_ERROR}]")
print("Z error:", "None" if Z_ERROR is None else f"q[{Z_ERROR}]")

## Circuit stages

In [ ]:
# Local syndrome inside a 3-qubit block: q0 -> 01, q1 -> 11, q2 -> 10
LOCAL_SYNDROME = {0: 0b01, 1: 0b11, 2: 0b10}
# Phase syndrome per block: block 1 -> 01, block 2 -> 11, block 3 -> 10
BLOCK_SYNDROME = {0: 0b01, 1: 0b11, 2: 0b10}


def new_shor_circuit():
    q = QuantumRegister(9, "q")
    a = QuantumRegister(8, "a")
    s = [ClassicalRegister(2, f"s{i}") for i in range(3)]
    p = ClassicalRegister(2, "p")
    return QuantumCircuit(q, a, *s, p), q, a, s, p


def encode(qc, q, alpha, beta):
    qc.prepare_state([alpha, beta], q[0])
    qc.barrier(label="Input State Preparation")

    # Phase-flip layer: spread over three blocks, then rotate to +/- basis
    qc.cx(q[0], q[3])
    qc.cx(q[0], q[6])
    for i in (0, 3, 6):
        qc.h(q[i])

    # Bit-flip layer: repeat each block leader twice
    for i in (0, 3, 6):
        qc.cx(q[i], q[i + 1])
        qc.cx(q[i], q[i + 2])

    qc.barrier(label="Shor Code Encoding")


def inject_errors(qc, q, x_error, z_error):
    qc.barrier(label="Before Error Injection")
    if x_error is not None:
        qc.x(q[x_error])
    if z_error is not None:
        qc.z(q[z_error])
    qc.barrier(label="After Error Injection")


def correct_bit_flips(qc, q, a, s):
    # Z_iZ_{i+1} and Z_{i+1}Z_{i+2} parities of each block
    for block in range(3):
        i = 3 * block
        qc.cx(q[i], a[2 * block])
        qc.cx(q[i + 1], a[2 * block])
        qc.cx(q[i + 1], a[2 * block + 1])
        qc.cx(q[i + 2], a[2 * block + 1])
    qc.barrier(label="Bit-Flip Syndrome Extraction")

    for block in range(3):
        qc.measure(a[2 * block], s[block][0])
        qc.measure(a[2 * block + 1], s[block][1])
    qc.barrier(label="Bit-Flip Syndrome Measurement")

    for block in range(3):
        for local, syndrome in LOCAL_SYNDROME.items():
            with qc.if_test((s[block], syndrome)):
                qc.x(q[3 * block + local])
    qc.barrier(label="Bit-Flip Error Correction")


def correct_phase_flip(qc, q, a, p):
    # In the Hadamard basis the X^{⊗6} stabilizers become Z-parities
    for i in range(9):
        qc.h(q[i])
    qc.barrier(label="Phase Basis Transformation")

    for i in range(6):                     # X0 X1 X2 X3 X4 X5
        qc.cx(q[i], a[6])
    qc.barrier(label="First Phase Syndrome Extraction")

    for i in range(3, 9):                  # X3 X4 X5 X6 X7 X8
        qc.cx(q[i], a[7])
    qc.barrier(label="Second Phase Syndrome Extraction")

    qc.measure(a[6], p[0])
    qc.measure(a[7], p[1])
    qc.barrier(label="Phase Syndrome Measurement")

    for i in range(9):
        qc.h(q[i])
    qc.barrier(label="Computational Basis Restoration")

    # A Z on any qubit of a block is equivalent; correct the block leader
    for block, syndrome in BLOCK_SYNDROME.items():
        with qc.if_test((p, syndrome)):
            qc.z(q[3 * block])
    qc.barrier(label="Phase-Flip Error Correction")


def decode(qc, q, a):
    for i in range(8):
        qc.reset(a[i])
    qc.barrier(label="Ancilla Reset")

    for i in (6, 3, 0):
        qc.cx(q[i], q[i + 2])
        qc.cx(q[i], q[i + 1])
    for i in (6, 3, 0):
        qc.h(q[i])
    qc.cx(q[0], q[6])
    qc.cx(q[0], q[3])
    qc.barrier(label="Shor Code Decoding")

    for i in range(1, 9):
        qc.reset(q[i])
    qc.barrier(label="Unused Data Qubits Reset")
    qc.barrier(label="Recovered State")


def shor_circuit(alpha, beta, x_error, z_error):
    qc, q, a, s, p = new_shor_circuit()
    encode(qc, q, alpha, beta)
    inject_errors(qc, q, x_error, z_error)
    correct_bit_flips(qc, q, a, s)
    correct_phase_flip(qc, q, a, p)
    decode(qc, q, a)
    return qc


def run_shor(alpha, beta, x_error, z_error):
    """Simulate the code; return (counts key 'p s2 s1 s0', recovered alpha, beta)."""
    qc = shor_circuit(alpha, beta, x_error, z_error)
    qc.save_statevector(label="final_state")

    result = backend.run(transpile(qc, backend, optimization_level=0), shots=1).result()
    key = next(iter(result.get_counts()))
    alpha_out, beta_out = qubit_state(result.data(0)["final_state"], qubit=0)

    return key, alpha_out, beta_out


def expected_syndromes(x_error, z_error):
    """Expected (bit-flip syndromes [s0, s1, s2], phase syndrome p)."""
    bit = [0, 0, 0]
    if x_error is not None:
        bit[x_error // 3] = LOCAL_SYNDROME[x_error % 3]
    phase = 0 if z_error is None else BLOCK_SYNDROME[z_error // 3]
    return bit, phase

## Stage-by-stage statevectors

In [ ]:
qc, q, a, s, p = new_shor_circuit()

encode(qc, q, alpha, beta)
print_sparse_state("STAGE 1: ENCODED QUANTUM STATE", get_snapshot(qc, "encoded", backend), n_bits=9)

inject_errors(qc, q, X_ERROR, Z_ERROR)
print_sparse_state("STAGE 2: QUANTUM STATE AFTER ERROR INJECTION",
                   get_snapshot(qc, "errored", backend), n_bits=9)

if X_ERROR is not None and X_ERROR == Z_ERROR:
    print("\nBoth errors act on the same physical qubit:")
    print("together they are equivalent to a Y error up to global phase.")

correct_bit_flips(qc, q, a, s)
correct_phase_flip(qc, q, a, p)
decode(qc, q, a)

save_circuit(qc, "circuit.png", idle_wires=False, style={"backgroundcolor": "white"});

## Syndromes and recovered state

In [ ]:
key, alpha_out, beta_out = run_shor(alpha, beta, X_ERROR, Z_ERROR)
p_bits, *s_bits = key.split()          # key order: p s2 s1 s0
s_bits = s_bits[::-1]                  # -> s0 s1 s2

bit_expected, phase_expected = expected_syndromes(X_ERROR, Z_ERROR)

print("SYNDROME RESULTS")
print("Measured (p s2 s1 s0):", key)
for i in range(3):
    print(f"Block {i + 1} (s{i}): measured {s_bits[i]}, expected {bit_expected[i]:02b}")
print(f"Phase (p)    : measured {p_bits}, expected {phase_expected:02b}")

print("\nERROR CORRECTION")
if X_ERROR is None:
    print("No bit-flip component detected.")
else:
    print(f"X error detected at q[{X_ERROR}] -> X applied to q[{X_ERROR}].")
if Z_ERROR is None:
    print("No phase-flip component detected.")
else:
    leader = 3 * (Z_ERROR // 3)
    print(f"Z error detected in the block of q[{Z_ERROR}] -> Z applied to q[{leader}]")
    print("(this corrects the equivalent phase error in the same Shor block).")

# Align global phase for a clearer side-by-side comparison
overlap = np.vdot([alpha, beta], [alpha_out, beta_out])
phase = np.exp(-1j * np.angle(overlap)) if abs(overlap) > 1e-12 else 1
f = fidelity([alpha, beta], [alpha_out, beta_out])

print("\nSHOR CODE RESULTS")
print("Original quantum state :", format_state(alpha, beta))
print("Recovered quantum state:", format_state(alpha_out * phase, beta_out * phase))
print("Fidelity:", round(f, 8))

## Verification — every combination of one X error and one Z error
10 × 10 = 100 runs (each error: none or qubit 0–8), including all $Y$ errors.

In [ ]:
failures = []
for x_error in (None, *range(9)):
    for z_error in (None, *range(9)):
        key, a_out, b_out = run_shor(alpha, beta, x_error, z_error)
        bit, phase = expected_syndromes(x_error, z_error)
        expected_key = " ".join([f"{phase:02b}"] + [f"{b:02b}" for b in reversed(bit)])

        if key != expected_key or not np.isclose(fidelity([alpha, beta], [a_out, b_out]), 1.0):
            failures.append((x_error, z_error, key, expected_key))

print(f"{100 - len(failures)} / 100 error combinations corrected with fidelity 1.")
assert not failures, failures
print("QUANTUM ERROR CORRECTION SUCCESSFUL for every single-qubit X, Z and Y error!")

In [ ]:
fig = qc.draw(output="mpl", fold=-1, idle_wires=False, style={"backgroundcolor": "white"})
display(fig)
plt.close(fig)